In [2]:
import pandas as pd
import torch
import pickle
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from transformers import AutoTokenizer, AutoModel

/Users/samirkadariya/opt/anaconda3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

# 1. Load the CSV (either from local file or raw GitHub URL)
#    If you want to fetch directly, use the raw URL below; 
#    otherwise replace with 'harmonized-system.csv'.
url = 'https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv'
df = pd.read_csv(url, dtype=str)


# get all of the rows that have level 2
chapters = df[df['level'] == str(2)]
headings = df[df['level'] == str(4)]
subheadings = df[df['level'] == str(6)]


chapters


,section,hscode,description,parent,level
0,I,01,Animals; live,TOTAL,2
41,I,02,Meat and edible meat offal,TOTAL,2
118,I,03,"Fish and crustaceans, molluscs and other aquat...",TOTAL,2
353,I,04,Dairy produce; birds' eggs; natural honey; edi...,TOTAL,2
397,I,05,Animal originated products; not elsewhere spec...,TOTAL,2
...,...,...,...,...,...
6733,XX,94,"Furniture; bedding, mattresses, mattress suppo...",TOTAL,2
6794,XX,95,"Toys, games and sports requisites; parts and a...",TOTAL,2
6840,XX,96,Miscellaneous manufactured articles,TOTAL,2
6910,XXI,97,Works of art; collectors' pieces and antiques,TOTAL,2


In [7]:
def setup_model_and_tokenizer():
    token = 'REDACTED_HF_TOKEN'
    model_name = 'sentence-transformers/all-mpnet-base-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
    model = AutoModel.from_pretrained(model_name, token=token)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    return tokenizer, model, device

In [8]:
def encode_text(texts, tokenizer, model, device, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        with torch.no_grad():
            model_output = model(**encoded_input)
        token_embeddings = model_output.last_hidden_state
        attention_mask = encoded_input['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

In [16]:
chapters

,section,hscode,description,parent,level
0,I,01,Animals; live,TOTAL,2
41,I,02,Meat and edible meat offal,TOTAL,2
118,I,03,"Fish and crustaceans, molluscs and other aquat...",TOTAL,2
353,I,04,Dairy produce; birds' eggs; natural honey; edi...,TOTAL,2
397,I,05,Animal originated products; not elsewhere spec...,TOTAL,2
...,...,...,...,...,...
6733,XX,94,"Furniture; bedding, mattresses, mattress suppo...",TOTAL,2
6794,XX,95,"Toys, games and sports requisites; parts and a...",TOTAL,2
6840,XX,96,Miscellaneous manufactured articles,TOTAL,2
6910,XXI,97,Works of art; collectors' pieces and antiques,TOTAL,2


In [17]:
# Ensure tokenizer, model, and device are available (from setup_model_and_tokenizer)
descriptions = chapters['description'].tolist()
tokenizer, model, device = setup_model_and_tokenizer()
embeddings = encode_text(descriptions, tokenizer, model, device)

In [18]:
# Save the embeddings to a file called chapters_embeddings.pkl
output_file = 'chapters_embeddings.pkl'
with open(output_file, 'wb') as f:
    pickle.dump(embeddings, f)

In [12]:
headings

,section,hscode,description,parent,level
1,I,0101,"Horses, asses, mules and hinnies; live",01,4
6,I,0102,Bovine animals; live,01,4
12,I,0103,Swine; live,01,4
16,I,0104,Sheep and goats; live,01,4
19,I,0105,"Poultry; live, fowls of the species Gallus dom...",01,4
...,...,...,...,...,...
6921,XXI,9703,"Sculptures and statuary; original, in any mate...",97,4
6924,XXI,9704,"Stamps, postage or revenue; stamp-postmarks, f...",97,4
6926,XXI,9705,Collections and collectors' pieces; of archaeo...,97,4
6933,XXI,9706,Antiques; of an age exceeding one hundred years,97,4
